In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv("../.env")
url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)

orders      = pd.read_sql("SELECT * FROM orders", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
customers   = pd.read_sql("SELECT * FROM customers", engine)
reviews     = pd.read_sql("SELECT * FROM order_reviews", engine)
print("Loaded from MySQL!")

Loaded from MySQL!


In [2]:
# ---------- Order-level features ----------
order_value = (order_items.groupby("order_id")
               .agg(items_total=("price", "sum"),
                    freight_total=("freight_value", "sum"),
                    item_count=("order_item_id", "count"))
               .reset_index())
order_value["total_order_value"] = order_value["items_total"] + order_value["freight_total"]
orders = orders.merge(order_value, on="order_id", how="left")

orders["delivery_days"] = (orders["order_delivered_customer_date"]
                           - orders["order_purchase_timestamp"]).dt.days
orders["delivery_delay_days"] = (orders["order_delivered_customer_date"]
                                 - orders["order_estimated_delivery_date"]).dt.days
orders["is_delayed"] = orders["delivery_delay_days"] > 0

# ---------- Customer features ----------
orders = orders.merge(
    customers[["customer_id", "customer_unique_id", "customer_state"]],
    on="customer_id", how="left"
)
valid_orders = orders[~orders["order_status"].isin(["canceled", "unavailable"])]

customer_features = (valid_orders.groupby("customer_unique_id")
    .agg(customer_order_count=("order_id", "nunique"),
         customer_total_spending=("total_order_value", "sum"))
    .reset_index())
customer_features["avg_order_value"] = (
    customer_features["customer_total_spending"] / customer_features["customer_order_count"]
)
customer_features["is_repeat_customer"] = customer_features["customer_order_count"] > 1

# ---------- Seller features ----------
seller_features = (order_items.groupby("seller_id")
    .agg(seller_revenue=("price", "sum"),
         seller_order_count=("order_id", "nunique"),
         seller_items_sold=("order_item_id", "count"))
    .reset_index())

# ---------- Save ----------
os.makedirs("../data/cleaned", exist_ok=True)
orders.to_csv("../data/cleaned/orders_features.csv", index=False)
customer_features.to_csv("../data/cleaned/customer_features.csv", index=False)
seller_features.to_csv("../data/cleaned/seller_features.csv", index=False)

print("Orders:", orders.shape)
print("Customers:", customer_features.shape)
print("Sellers:", seller_features.shape)
print("Features saved!")

Orders: (99441, 17)
Customers: (94990, 5)
Sellers: (3095, 4)
Features saved!
